0. Project setup
1. Load datasets using data_provider/data_loader.py
2. Inspect standardized data
3. Create binary classification dataset
4. Train TF-IDF + LinearSVC baseline
5. Evaluate binary performance
6. Create multi-class / model-attribution dataset
7. Train multi-class SVM
8. Cross-dataset evaluation
9. Error analysis
10. Save results and conclusions

# SVM Baseline for AI-Generated Text Detection #

## 0. Goal ##
This notebook implements TF-IDF + LinearSVC for: 
- binary human-vs-AI detection
- AI model attribution
- cross-dataset generalization

## 1. Smoke-test the data loader ##

In [ ]:
# Setup
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "svm"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

In [4]:
# Import loader
import pandas as pd
import numpy as np

from data_provider.data_loader import (
    load_dataset,
    make_binary_dataset,
    make_multiclass_dataset,
)

In [5]:
df = load_dataset(foldername=DATA_DIR)
df.head()

,text,binary_label,source_model,dataset_name,domain,original_label
0,Cars. Cars have been around since they became ...,human,human,ai_vs_human_text,unknown,0.0
1,Transportation is a large necessity in most co...,human,human,ai_vs_human_text,unknown,0.0
2,"""America's love affair with it's vehicles seem...",human,human,ai_vs_human_text,unknown,0.0
3,How often do you ride in a car? Do you drive a...,human,human,ai_vs_human_text,unknown,0.0
4,Cars are a wonderful thing. They are perhaps o...,human,human,ai_vs_human_text,unknown,0.0


In [6]:
# Sanity checks
print(df.shape)
print(df.columns)

display(df.head())
display(df["dataset_name"].value_counts())
display(df["binary_label"].value_counts())
display(df["source_model"].value_counts())

(478799, 6)
Index(['text', 'binary_label', 'source_model', 'dataset_name', 'domain',
       'original_label'],
      dtype='object')


,text,binary_label,source_model,dataset_name,domain,original_label
0,Cars. Cars have been around since they became ...,human,human,ai_vs_human_text,unknown,0.0
1,Transportation is a large necessity in most co...,human,human,ai_vs_human_text,unknown,0.0
2,"""America's love affair with it's vehicles seem...",human,human,ai_vs_human_text,unknown,0.0
3,How often do you ride in a car? Do you drive a...,human,human,ai_vs_human_text,unknown,0.0
4,Cars are a wonderful thing. They are perhaps o...,human,human,ai_vs_human_text,unknown,0.0


dataset_name
ai_vs_human_text             465199
ai_text_detection_dataset     10129
daigt_v2_train                 2971
multi_model_detection           500
Name: count, dtype: Int64

binary_label
human    285597
ai       193202
Name: count, dtype: Int64

source_model
human                                 285597
unknown_ai                            189912
llama2_chat                             1754
darragh_claude_v6                        406
NousResearch/Llama-2-7b-chat-hf          297
palm-text-bison1                         196
cohere-command                           161
mistralai/Mistral-7B-Instruct-v0.1       156
Gemini-2_Pro                             118
GPT-5_Preview                            107
Claude-4_Opus                             95
Name: count, dtype: Int64

## 2. Add notebook-level audit checks ##

In [7]:
# Missing values
df.isna().sum()

# Check for duplicate text after loading
df["text"].duplicated().sum()

# Check text lengths
df["num_chars"] = df["text"].str.len()
df["num_words"] = df["text"].str.split().str.len()

display(df.groupby("binary_label")[["num_chars", "num_words"]].describe())
display(df.groupby("dataset_name")[["num_chars", "num_words"]].describe())

num_chars                                                  \
                 count         mean          std   min     25%     50%   
binary_label                                                             
ai            193202.0  2121.331798   785.874158   1.0  1653.0  2049.0   
human         285597.0  2356.851448  1088.997213  76.0  1538.0  2157.0   

                             num_words                                       \
                 75%     max     count        mean         std   min    25%   
binary_label                                                                  
ai            2482.0  8857.0  193202.0  344.302621  117.160317   1.0  274.0   
human         2928.0  9180.0  285597.0  423.000074  188.420512  13.0  281.0   

                                    
                50%    75%     max  
binary_label                        
ai            338.0  404.0  1238.0  
human         390.0  522.0  1668.0

num_chars                                          \
                              count         mean         std    min     25%   
dataset_name                                                                  
ai_text_detection_dataset   10129.0  2014.998420  810.020969    7.0  1373.0   
ai_vs_human_text           465199.0  2271.181742  988.220716    1.0  1587.0   
daigt_v2_train               2971.0  1970.788960  439.945313  508.0  1719.5   
multi_model_detection         500.0   277.282000   84.772964   96.0   220.0   

                                                   num_words              \
                              50%      75%     max     count        mean   
dataset_name                                                               
ai_text_detection_dataset  2099.0  2573.00  5990.0   10129.0  331.749729   
ai_vs_human_text           2103.0  2722.00  9180.0  465199.0  393.266780   
daigt_v2_train             1879.0  2101.00  4445.0    2971.0  337.524403   
multi_model_detection       268.0   326.25   608.0     500.0   34.232000   

                                                                            
                                  std   min     25%    50%     75%     max  
dataset_name                                                                
ai_text_detection_dataset  127.049050   2.0  236.00  344.0  415.00   998.0  
ai_vs_human_text           168.559918   1.0  278.00  362.0  471.00  1668.0  
daigt_v2_train              68.720100   4.0  294.00  326.0  364.00   713.0  
multi_model_detection        9.448212  13.0   27.75   34.0   40.25    64.0

In [8]:
pd.set_option("display.max_colwidth", 300)

display(df.sample(10, random_state=RANDOM_STATE))

,text,binary_label,source_model,dataset_name,domain,original_label,num_chars,num_words
477305,"The Face on Mars is a natural landform. This is evidenced by the fact that the Face has been eroded over time by the Martian elements. The Face is not a perfect symmetrical structure, as would be expected if it were created by aliens. The Face is also not aligned with the Martian grid, which is ...",ai,unknown_ai,ai_text_detection_dataset,unknown,1,345,64
434853,"The electoral college process consist of the selection of the elector, the meeting of the electors where they vote for president and vice president in a way to let us the people choose to keep our country safe and to help our community. So I think we should keep the electoral college.\n\nUnder t...",human,human,ai_vs_human_text,unknown,0.0,1583,283
73707,"Dear Senator, I know that you have many issues to think about and have a lot of decisions to make, but I think it is the subject of the Electoral College is a very important subject for you to ponder. The Electoral College needs to be changed, we need to vote for the presidency with the popular ...",human,human,ai_vs_human_text,unknown,0.0,3825,664
322557,"There is no definitive answer to this question as it depends on the individual. However, in general, young people are more likely to enjoy life than older people are.\n\nThere are several reasons for this. Eirstly, young people are typically more optimistic and energetic than older people. They ...",ai,unknown_ai,ai_vs_human_text,unknown,1.0,1054,174
66622,Many schools have established partnerships with companies to provide students with opportunities to explore various aspects of business. These partnerships are beneficial for students who have goals of starting their own businesses or who want to gain experience in the business world. By partner...,ai,unknown_ai,ai_vs_human_text,unknown,1.0,1479,220
109278,"My principle has decided that she wants everyone to participate in an extracurricular activity. In CY opinion, I believe it's the best for students to participate in school activity. It will be a great way to have fun, cake new friends, and get some exercise.\n\nMaking new friends is not always ...",human,human,ai_vs_human_text,unknown,0.0,3271,617
32988,The Electoral College is a system in the United States that is used to elect the President and Vice President of the country every four years. The question of whether it works or not is a subject of debate and can be analyzed in various ways.\n\nOne way to examine the efficacy of the Electoral C...,ai,unknown_ai,ai_vs_human_text,unknown,1.0,1940,320
62526,"In the world of education, there have been many arguments about which way of learning could benefit the most people. Some agree that the system we have now is the pinnacle of what we can do to help younger generations learn. However, there is definitely more we could do so the next firemen or po...",human,human,ai_vs_human_text,unknown,0.0,3279,586
313172,"The Face on Mars is just a natural landform. Why I say that because, aliens are not real, and they can't create the landforms that were seen. If there were aliens then it would have been more information found on NASA. If no human or thing can't breath in space an alien shouldn't be able to brea...",human,human,ai_vs_human_text,unknown,0.0,1045,197
436063,"Limit Of Car Usage\n\nWhen you think about limiting car usage you might think ""Well how am I supposed to get there. There's no way I'm able to get to my Destination without a car."" Yes, I'm pretty sure we all would'NT want to walk five thousand miles to visit a friend three hours away, but here ...",human,human,ai_vs_human_text,unknown,0.0,3003,591


## 3. Save processed snapshot ##

In [9]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_DIR / "standardized_all.csv", index=False)

## 4. Binary SVM experiment ##

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [11]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

def evaluate_model(model, X_test, y_test, task_name):
    y_pred = model.predict(X_test)

    print(f"=== {task_name} ===")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))

    results = {
        "task": task_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro"),
        "weighted_f1": f1_score(y_test, y_pred, average="weighted"),
        "num_test": len(y_test),
    }

    return results, y_pred

In [12]:
binary_df = make_binary_dataset(df)

X = binary_df["text"]
y = binary_df["binary_label"]

print(binary_df.shape)
print(y.value_counts())

(478799, 6)
binary_label
human    285597
ai       193202
Name: count, dtype: Int64


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

In [14]:
binary_svm = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=100_000,
    )),
    ("clf", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

In [15]:
binary_svm.fit(X_train, y_train)

y_pred = binary_svm.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

          ai       1.00      1.00      1.00     38640
       human       1.00      1.00      1.00     57120

    accuracy                           1.00     95760
   macro avg       1.00      1.00      1.00     95760
weighted avg       1.00      1.00      1.00     95760

[[38599    41]
 [   13 57107]]


In [16]:
binary_results = {
    "task": "binary_human_vs_ai",
    "model": "tfidf_linearsvc",
    "accuracy": accuracy_score(y_test, y_pred),
    "macro_f1": f1_score(y_test, y_pred, average="macro"),
    "weighted_f1": f1_score(y_test, y_pred, average="weighted"),
    "num_train": len(X_train),
    "num_test": len(X_test),
}

pd.DataFrame([binary_results]).to_csv(
    RESULTS_DIR / "binary_results.csv",
    index=False,
)

binary_results

{'task': 'binary_human_vs_ai',
 'model': 'tfidf_linearsvc',
 'accuracy': 0.9994360902255639,
 'macro_f1': 0.999414207861153,
 'weighted_f1': 0.9994360571206254,
 'num_train': 383039,
 'num_test': 95760}

In [17]:
error_df = pd.DataFrame({
    "text": X_test,
    "true_label": y_test,
    "predicted_label": y_pred,
})

errors = error_df[error_df["true_label"] != error_df["predicted_label"]]

print(errors.shape)
display(errors.sample(min(10, len(errors)), random_state=RANDOM_STATE))

(54, 3)


,text,true_label,predicted_label
473976,"Have you ever aspired to join NASA or become a scientist? Scientists have the opportunity to explore various parts of the world, including outer space. Have you ever had the desire to journey to Mars, also known as the ""Red Planet?"" Mars has an extensive history, and some NASA scientists claim t...",ai,human
254741,"NO MORE LIMITING!\n\nSome people believe that students should be required TT take a music, a drama, try an art class. But should students be forcibly required TT take a music, a drama, try an art class?\n\nStudents should NTT be required TT take a music try art classes, and make these classes as...",ai,human
351203,"Thomas Jefferson once said, ""Determine never ZO be idle...IZ is wonderful how much may be done if we are always doing."" Being active like pursuing career goals and helping others and being inactive can lead ZO a LOZ of health problems.\n\nDo people accomplish more if they are pursuing their own ...",ai,human
174233,"Have you ever made a awful mistake and think you can never come back from it? You can always come back from a mistake. Duke Ellington the famous jazz legend always said ""a problem is a chance to do your best because you learn from your mistakes.\n\nIf you mess up and make a mistake you will lear...",human,ai
473248,"No, I don't believe we should advocate for autonomous vehicles as they pose hazardous situations. \n\nIn such scenarios, the driver has no control whatsoever, rendering their presence in the vehicle pointless. This could potentially be very dangerous, as instead of the individual controlling the...",ai,human
166522,"First impressions are what I learn from many people. People have many impressions and yes, it is hard to change.\n\nImpressions are special to some people, but it also could affect people in certain ways. Ways where your whole behavior might change or attitude or maybe your whole person. When we...",ai,human
474024,"Have you ever fantasized about sitting back and unwinding while your vehicle does all the driving? Have you ever pondered the potential risks of such a scenario? I am of the opinion that semi or fully self-driving cars pose a threat to the world. These cars are not entirely autonomous, it's uncl...",ai,human
473285,"Dear Principal,\n\nI believe both of your proposed policies have their merits, however, to prevent students from getting into trouble, I feel that your first policy, which allows students to use their phones during lunch and free periods, is a better option. This policy also requires students to...",ai,human
473144,"Hello, I am known as Generic_Name and I am a student at Generic_School. Our principal, also named Generic_Name, has made a decision that every student must engage in at least one extracurricular activity. I strongly oppose this decision and believe that Generic_Name is mistaken. I believe that t...",ai,human
474739,"I argue in favor of keeping the Electoral College because it is a technicality that should be taken into consideration. We as a society are built around the idea ofvotes, and while it is possible to get by with only a popular vote, it is not possible to get by with only a popular vote. The Elect...",ai,human


In [18]:
binary_train, binary_test = train_test_split(
    binary_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=binary_df["binary_label"],
)

X_train = binary_train["text"]
y_train = binary_train["binary_label"]

X_test = binary_test["text"]
y_test = binary_test["binary_label"]

In [19]:
binary_svm.fit(X_train, y_train)
y_pred = binary_svm.predict(X_test)

analysis_df = binary_test.copy()
analysis_df["predicted_label"] = y_pred
analysis_df["correct"] = analysis_df["binary_label"] == analysis_df["predicted_label"]

display(analysis_df.groupby("dataset_name")["correct"].mean())
display(analysis_df.groupby("source_model")["correct"].mean())
display(analysis_df.groupby("binary_label")["correct"].mean())

display(
    analysis_df[~analysis_df["correct"]][
        ["text", "binary_label", "predicted_label", "dataset_name", "source_model", "domain"]
    ].sample(10, random_state=RANDOM_STATE)
)

dataset_name
ai_text_detection_dataset    0.989071
ai_vs_human_text             0.999688
daigt_v2_train                    1.0
multi_model_detection        0.968085
Name: correct, dtype: Float64

source_model
Claude-4_Opus                              1.0
GPT-5_Preview                              1.0
Gemini-2_Pro                               1.0
NousResearch/Llama-2-7b-chat-hf            1.0
cohere-command                             1.0
darragh_claude_v6                          1.0
human                                 0.999772
llama2_chat                                1.0
mistralai/Mistral-7B-Instruct-v0.1         1.0
palm-text-bison1                           1.0
unknown_ai                             0.99892
Name: correct, dtype: Float64

binary_label
ai       0.998939
human    0.999772
Name: correct, dtype: Float64

,text,binary_label,predicted_label,dataset_name,source_model,domain
473976,"Have you ever aspired to join NASA or become a scientist? Scientists have the opportunity to explore various parts of the world, including outer space. Have you ever had the desire to journey to Mars, also known as the ""Red Planet?"" Mars has an extensive history, and some NASA scientists claim t...",ai,human,ai_text_detection_dataset,unknown_ai,unknown
254741,"NO MORE LIMITING!\n\nSome people believe that students should be required TT take a music, a drama, try an art class. But should students be forcibly required TT take a music, a drama, try an art class?\n\nStudents should NTT be required TT take a music try art classes, and make these classes as...",ai,human,ai_vs_human_text,unknown_ai,unknown
351203,"Thomas Jefferson once said, ""Determine never ZO be idle...IZ is wonderful how much may be done if we are always doing."" Being active like pursuing career goals and helping others and being inactive can lead ZO a LOZ of health problems.\n\nDo people accomplish more if they are pursuing their own ...",ai,human,ai_vs_human_text,unknown_ai,unknown
174233,"Have you ever made a awful mistake and think you can never come back from it? You can always come back from a mistake. Duke Ellington the famous jazz legend always said ""a problem is a chance to do your best because you learn from your mistakes.\n\nIf you mess up and make a mistake you will lear...",human,ai,ai_vs_human_text,human,unknown
473248,"No, I don't believe we should advocate for autonomous vehicles as they pose hazardous situations. \n\nIn such scenarios, the driver has no control whatsoever, rendering their presence in the vehicle pointless. This could potentially be very dangerous, as instead of the individual controlling the...",ai,human,ai_text_detection_dataset,unknown_ai,unknown
166522,"First impressions are what I learn from many people. People have many impressions and yes, it is hard to change.\n\nImpressions are special to some people, but it also could affect people in certain ways. Ways where your whole behavior might change or attitude or maybe your whole person. When we...",ai,human,ai_vs_human_text,unknown_ai,unknown
474024,"Have you ever fantasized about sitting back and unwinding while your vehicle does all the driving? Have you ever pondered the potential risks of such a scenario? I am of the opinion that semi or fully self-driving cars pose a threat to the world. These cars are not entirely autonomous, it's uncl...",ai,human,ai_text_detection_dataset,unknown_ai,unknown
473285,"Dear Principal,\n\nI believe both of your proposed policies have their merits, however, to prevent students from getting into trouble, I feel that your first policy, which allows students to use their phones during lunch and free periods, is a better option. This policy also requires students to...",ai,human,ai_text_detection_dataset,unknown_ai,unknown
473144,"Hello, I am known as Generic_Name and I am a student at Generic_School. Our principal, also named Generic_Name, has made a decision that every student must engage in at least one extracurricular activity. I strongly oppose this decision and believe that Generic_Name is mistaken. I believe that t...",ai,human,ai_text_detection_dataset,unknown_ai,unknown
474739,"I argue in favor of keeping the Electoral College because it is a technicality that should be taken into consideration. We as a society are built around the idea ofvotes, and while it is possible to get by with only a popular vote, it is not possible to get by with only a popular vote. The Elect...",ai,human,ai_text_detection_dataset,unknown_ai,unknown


Note on the above result: this is on the unbalanced, combined dataset. As you can see from the section 1 sanity check, most of the training/testing examples come from one dataset accounting for 97% of the combined dataset. 

## 5. Multi-class SVM experiment ##

In [ ]:
multi_df = make_multiclass_dataset(df, min_examples_per_class=100)

# For AI model attribution, remove human examples
multi_ai_df = multi_df[multi_df["binary_label"] == "ai"].copy()

print(multi_ai_df.shape)
print(multi_ai_df["source_model"].value_counts())

In [ ]:
X = multi_ai_df["text"]
y = multi_ai_df["source_model"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

In [ ]:
multi_svm = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=100_000,
    )),
    ("clf", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

multi_svm.fit(X_train, y_train)
y_pred = multi_svm.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
multi_results = {
    "task": "ai_model_attribution",
    "model": "tfidf_linearsvc",
    "accuracy": accuracy_score(y_test, y_pred),
    "macro_f1": f1_score(y_test, y_pred, average="macro"),
    "weighted_f1": f1_score(y_test, y_pred, average="weighted"),
    "num_train": len(X_train),
    "num_test": len(X_test),
    "num_classes": y.nunique(),
}

pd.DataFrame([multi_results]).to_csv(
    RESULTS_DIR / "multiclass_ai_model_results.csv",
    index=False,
)

multi_results

## 6. Cross-dataset evaluation ##

In [ ]:
df["dataset_name"].value_counts()

In [ ]:
train_dataset = "daigt_v2_train"
test_dataset = "ai_vs_human_text"

train_df = binary_df[binary_df["dataset_name"] == train_dataset].copy()
test_df = binary_df[binary_df["dataset_name"] == test_dataset].copy()

print(train_df["binary_label"].value_counts())
print(test_df["binary_label"].value_counts())

In [ ]:
X_train = train_df["text"]
y_train = train_df["binary_label"]

X_test = test_df["text"]
y_test = test_df["binary_label"]

cross_svm = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=100_000,
    )),
    ("clf", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

cross_svm.fit(X_train, y_train)
y_pred = cross_svm.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))